# HPMA + SAM 3.1 trên Kaggle (2 × NVIDIA T4)

Notebook này là bản Kaggle của `train_colab_l4.ipynb`. Code, dataset đã giải nén, SAM checkpoint, memory bank, log và model checkpoints được đặt dưới `/kaggle/working/StrokeSAM` để Kaggle có thể lưu artifacts khi tạo notebook version.

**Hai GPU có tổng VRAM danh nghĩa khoảng 30 GB nhưng VRAM không được gộp thành một vùng nhớ duy nhất.** DataParallel sao chép model lên từng GPU; mỗi T4 vẫn bị giới hạn bởi 15 GB riêng. Cấu hình khởi đầu dùng global batch 2 (mỗi GPU nhận 1 mẫu), gradient accumulation 8 (effective batch 16). Đừng đặt batch 29 GB; hãy đo từng GPU rồi mới tăng global batch lên 4 nếu còn dư bộ nhớ.

Chọn **Save Version → Save & Run All (Commit)**, rồi trong Advanced Settings chọn **Run with GPU for all sessions** để commit có thể dùng hai T4. Kaggle chạy bản sao sạch; cell setup cài dependencies trước các import cần chúng, nên không cần restart kernel thủ công. Bật Internet và chọn Accelerator **GPU T4 x2** trong Notebook Settings. Kaggle lưu tối đa 20 GB output của notebook trong `/kaggle/working`; kiểm tra dung lượng ở cell cuối trước khi chọn **Save Version**. [Kaggle Notebook documentation](https://www.kaggle.com/docs/notebooks)

## 1. Thiết lập môi trường Python sạch và xác nhận hai GPU T4

## 2. Clone project vào `/kaggle/working`

In [ ]:
from pathlib import Path
import subprocess
import sys

WORKING_DIR = Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'StrokeSAM'

if REPO_DIR.exists():
    # Training setup rewrites these tracked YAMLs; restore them before updating code.
    subprocess.run(
        [
            'git', 'restore', '--worktree',
            'configs/base_config.yaml',
            'configs/isles2022.yaml',
        ],
        cwd=REPO_DIR,
        check=True,
    )
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/hoangtung386/StrokeSAM.git', str(REPO_DIR)],
        check=True,
    )

print(f"Project: {REPO_DIR}")

In [ ]:
required_paths = [
    REPO_DIR / 'models' / 'hpma_heads.py',
    REPO_DIR / 'notebooks' / 'train_kaggle_2xt4.ipynb',
]

missing = [str(path) for path in required_paths if not path.is_file()]
train_source = (REPO_DIR / 'train.py').read_text(encoding='utf-8')
build_source = (REPO_DIR / 'build_memory.py').read_text(encoding='utf-8')

if 'use_data_parallel' not in train_source or 'use_data_parallel' not in build_source:
    missing.append('DataParallel changes in train.py/build_memory.py')

if missing:
    raise RuntimeError(
        'GitHub project revision is missing Kaggle support changes: '
        + ', '.join(missing)
        + '. Commit and push the current project changes before running this notebook.'
    )

## 3. Dependencies được cài trước khi Kaggle khởi động kernel

Lỗi trước đó do cài SAM3 kéo NumPy xuống 1.26.4 rồi cell sau thay bằng 2.2.6 ngay trong kernel; NumPy native modules bị trộn phiên bản. Hãy cấu hình một lần trong **Add-ons → Dependency Manager** để các wheel được cài trước khi Save & Run All mở kernel. Kaggle hỗ trợ Dependency Manager trong commit.

Thêm các lệnh sau vào Dependency Manager (mỗi dòng là một lệnh):

```text
pip install numpy==2.2.6 scipy==1.15.3 scikit-learn==1.6.1 opencv-python-headless==4.14.0.94 nibabel pyyaml tqdm matplotlib requests huggingface_hub wandb timm ftfy==6.1.1 regex iopath typing_extensions
pip install --no-deps git+https://github.com/facebookresearch/sam3.git
```

SAM3 metadata hiện yêu cầu NumPy < 2; trên Kaggle Python 3.13, dùng `--no-deps` để pip không hạ NumPy trong phiên. Cell dưới xác minh NumPy/SciPy/SAM3 import được. Nếu import SAM3 không qua, dừng ở đây và xem log cài Dependency Manager trước khi chạy train.

In [ ]:
import subprocess
import sys
import numpy
import scipy
import scipy.ndimage
import sklearn
import torch

from sam3.model_builder import build_sam3_image_model

if sys.version_info < (3, 12):
    raise RuntimeError(f"SAM 3 requires Python >= 3.12; runtime is {sys.version.split()[0]}.")

# Register the checked-out project without allowing pip to replace Kaggle's runtime packages.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'],
    cwd=REPO_DIR,
    check=True,
)

if numpy.__version__ != '2.2.6' or scipy.__version__ != '1.15.3':
    raise RuntimeError(
        'Dependency Manager chưa cài đúng NumPy 2.2.6 / SciPy 1.15.3 trước khi kernel khởi động.'
    )

print(
    f"NumPy {numpy.__version__}; SciPy {scipy.__version__}; "
    f"scikit-learn {sklearn.__version__}; PyTorch {torch.__version__}"
)

print(f"SAM3 import OK: {build_sam3_image_model.__module__}")

In [ ]:
if sys.version_info < (3, 12):
    raise RuntimeError(f"SAM 3 yêu cầu Python >= 3.12; runtime là {sys.version.split()[0]}.")
if tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) < (2, 7):
    raise RuntimeError(f"SAM 3 yêu cầu PyTorch >= 2.7; đang dùng {torch.__version__}.")
if torch.version.cuda is None or tuple(map(int, torch.version.cuda.split('.')[:2])) < (12, 6):
    raise RuntimeError(f"SAM 3 yêu cầu CUDA >= 12.6; đang dùng {torch.version.cuda}.")


print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError(
        "Hãy bật Accelerator = GPU T4 x2 trong Notebook Settings rồi chạy lại."
    )

for index in range(2):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {props.name}, {props.total_memory / 2**30:.2f} GiB")
    if "T4" not in props.name.upper():
        print("Cảnh báo: GPU này không phải T4; hãy kiểm tra accelerator đã chọn.")

torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = True

## 4. Tải ISLES-2022 từ Zenodo và checkpoint SAM 3.1

Dataset ZIP được tải vào `/kaggle/working`, giải nén vào `StrokeSAM/dataset/ISLES-2022`, kiểm tra cấu trúc rồi mới xóa ZIP. Nếu cell bị ngắt giữa chừng, lần chạy sau thử resume bằng HTTP Range.

Checkpoint SAM 3.1 là gated trên Hugging Face. Trước tiên hãy chấp thuận quyền truy cập model `facebook/sam3.1`, rồi tạo Kaggle Secret tên `HF_TOKEN` chứa Hugging Face token. Checkpoint sau khi tải nằm trong `/kaggle/working/StrokeSAM/checkpoints` và sẽ được đưa vào output version.

In [ ]:
import shutil
import time
import requests
from zipfile import ZipFile, BadZipFile
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("WANDB_API_KEY")

login(token=secret_value_0, add_to_git_credential=False)
wandb.login(key=secret_value_1)

In [ ]:
DATASET_URL = 'https://zenodo.org/records/7960856/files/ISLES-2022.zip?download=1'
DATASET_ZIP = WORKING_DIR / 'ISLES-2022.zip'
DATASET_DIR = REPO_DIR / 'dataset' / 'ISLES-2022'
STAGING_DIR = REPO_DIR / 'dataset' / '.ISLES-2022-extract'
CHECKPOINT_DIR = REPO_DIR / 'checkpoints'
SAM_CHECKPOINT = CHECKPOINT_DIR / 'sam3.1.pt'
OFFICIAL_SAM3_CHECKPOINT = CHECKPOINT_DIR / 'sam3.1_multiplex.pt'


def has_isles_layout(root):
    dwi_file = next(root.glob('sub-*/ses-*/dwi/*_dwi.nii.gz'), None)
    return dwi_file is not None and (root / 'derivatives').is_dir()


def download_dataset(url, destination, attempts=6):
    for attempt in range(1, attempts + 1):
        offset = destination.stat().st_size if destination.exists() else 0
        headers = {'Range': f'bytes={offset}-'} if offset else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as response:
                response.raise_for_status()
                content_range = response.headers.get('Content-Range', '')
                resumed = bool(offset and response.status_code == 206)
                if resumed:
                    if not content_range.startswith(f'bytes {offset}-'):
                        raise IOError(f"Content-Range không khớp: {content_range}")
                    total = int(content_range.rsplit('/', 1)[1])
                    mode = 'ab'
                else:
                    offset = 0
                    mode = 'wb'
                    length = response.headers.get('Content-Length')
                    total = int(length) if length else None
                with destination.open(mode) as output, tqdm(
                    total=total,
                    initial=offset,
                    unit='B',
                    unit_scale=True,
                    desc=f'ISLES-2022.zip ({attempt}/{attempts})',
                ) as progress:
                    for chunk in response.iter_content(8 * 1024 * 1024):
                        if chunk:
                            output.write(chunk)
                            progress.update(len(chunk))
            actual = destination.stat().st_size
            if total is not None and actual != total:
                raise IOError(f"Tải thiếu dữ liệu: {actual:,}/{total:,} bytes.")
            with ZipFile(destination) as archive:
                if not archive.namelist():
                    raise BadZipFile('ZIP rỗng.')
            return
        except BadZipFile as error:
            destination.unlink(missing_ok=True)
            if attempt == attempts:
                raise RuntimeError('Zenodo liên tục trả ZIP hỏng.') from error
        except (requests.RequestException, IOError) as error:
            if attempt == attempts:
                raise RuntimeError(f"Không tải xong dataset sau {attempts} lần: {error}") from error
            print(f"Tải bị ngắt; sẽ thử tiếp từ byte đã nhận: {error}")
            time.sleep(min(2 ** attempt, 30))

In [ ]:
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
if not has_isles_layout(DATASET_DIR):
    if STAGING_DIR.exists():
        shutil.rmtree(STAGING_DIR)
    
    STAGING_DIR.mkdir(parents=True, exist_ok=True)
    download_dataset(DATASET_URL, DATASET_ZIP)
    print('Đang giải nén ISLES-2022...')
    with ZipFile(DATASET_ZIP) as archive:
        archive.extractall(STAGING_DIR)
    dwi_file = next(STAGING_DIR.rglob('*_dwi.nii.gz'), None)
    if dwi_file is None:
        raise FileNotFoundError('Không tìm thấy *_dwi.nii.gz trong ZIP.')
    
    extracted_root = dwi_file.parents[3]
    if not (extracted_root / 'derivatives').is_dir():
        raise FileNotFoundError(f"Thiếu derivatives trong {extracted_root}")
    
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    
    shutil.move(str(extracted_root), str(DATASET_DIR))
    shutil.rmtree(STAGING_DIR, ignore_errors=True)
    
    if not has_isles_layout(DATASET_DIR):
        raise FileNotFoundError(f"Cấu trúc dataset không hợp lệ: {DATASET_DIR}")
    
    DATASET_ZIP.unlink()
else:
    print(f"Dùng lại dataset đã giải nén: {DATASET_DIR}")
    DATASET_ZIP.unlink(missing_ok=True)

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if not OFFICIAL_SAM3_CHECKPOINT.is_file():
    hf_hub_download(
        repo_id='facebook/sam3.1',
        filename='sam3.1_multiplex.pt',
        local_dir=str(CHECKPOINT_DIR),
    )

In [ ]:
if not OFFICIAL_SAM3_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Không tải được checkpoint: {OFFICIAL_SAM3_CHECKPOINT}")

if SAM_CHECKPOINT.is_symlink() or SAM_CHECKPOINT.exists():
    SAM_CHECKPOINT.unlink()

SAM_CHECKPOINT.symlink_to(OFFICIAL_SAM3_CHECKPOINT.name)

print(f"Dataset: {DATASET_DIR}")
print(f"Checkpoint: {SAM_CHECKPOINT}")
print(f"Found {sum(1 for _ in DATASET_DIR.glob('sub-*/ses-*/dwi/*_dwi.nii.gz'))} cases")

## 5. Cấu hình split 80/10/10 và multi-GPU

In [ ]:
import yaml

BASE_CONFIG = REPO_DIR / 'configs' / 'base_config.yaml'
DATA_CONFIG = REPO_DIR / 'configs' / 'isles2022.yaml'

with BASE_CONFIG.open(encoding='utf-8') as stream:
    base_cfg = yaml.safe_load(stream)
with DATA_CONFIG.open(encoding='utf-8') as stream:
    data_cfg = yaml.safe_load(stream)

data_cfg['dataset']['data_root'] = str(DATASET_DIR)
data_cfg['dataset']['validation_fraction'] = 0.10
data_cfg['dataset']['test_fraction'] = 0.10
base_cfg['system']['device'] = 'cuda:0'
base_cfg['system']['use_data_parallel'] = True
base_cfg['system']['num_workers'] = 2
base_cfg['system']['seed'] = 42
base_cfg['model']['sam3_checkpoint'] = str(SAM_CHECKPOINT)
base_cfg['training']['epochs'] = 100
base_cfg['training']['batch_size'] = 2  # global batch: 1 sample per T4
base_cfg['training']['gradient_accumulation_steps'] = 8
base_cfg['training']['mixed_precision'] = True

with BASE_CONFIG.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(base_cfg, stream, sort_keys=False)
with DATA_CONFIG.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(data_cfg, stream, sort_keys=False)
print('DataParallel: cuda:0 + cuda:1; per-GPU batch=1; effective batch=16')

## 6. Kiểm tra split và xem preprocessing

In [ ]:
import sys
import matplotlib.pyplot as plt

# Jupyter's working directory may not be the cloned repository.
sys.path.insert(0, str(REPO_DIR))

try:
    import scipy.ndimage
except ImportError as error:
    raise RuntimeError(
        "SciPy ndimage vẫn không import được sau khi cài lại scientific stack. "
        "Kiểm tra phiên bản được in ở cell cài đặt NumPy/SciPy và chạy lại toàn bộ notebook."
    ) from error

from data.factory import build_dataset

common = dict(
    dataset_config=data_cfg['dataset'],
    target_size=tuple(base_cfg['data']['input_resolution']),
    seed=base_cfg['system']['seed'],
    mean=tuple(base_cfg['data']['mean']),
    std=tuple(base_cfg['data']['std']),
)

datasets = {name: build_dataset(split=name, **common) for name in ('train', 'val', 'test')}
splits = {name: set(ds.case_splits[name]) for name, ds in datasets.items()}

assert not (splits['train'] & splits['val'])
assert not (splits['train'] & splits['test'])
assert not (splits['val'] & splits['test'])

for name, ids in splits.items():
    print(f"{name}: {len(ids)} cases, {len(datasets[name])} slices")

In [ ]:
import numpy as np

sample = None
for index in range(min(512, len(datasets['train']))):
    candidate = datasets['train'][index]
    if candidate['mask'].any():
        sample = candidate
        break
if sample is None:
    raise RuntimeError('Không tìm thấy lát cắt train có tổn thương trong 512 mẫu đầu.')

image = sample['image'].numpy()
mask = sample['mask'][0].numpy()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, channel, title in zip(axes[:3], image, ('DWI', 'ADC', 'FLAIR')):
    valid = np.isfinite(channel) & (np.abs(channel) > 1e-6)
    if not valid.any():
        raise ValueError(f'{title} không có voxel khác 0 để hiển thị.')
    
    low, high = np.percentile(channel[valid], [1, 99])
    ax.imshow(np.ma.masked_where(~valid, channel), cmap='gray', vmin=low, vmax=high)
    ax.set_title(title)
    ax.axis('off')

valid_dwi = np.isfinite(image[0]) & (np.abs(image[0]) > 1e-6)
low, high = np.percentile(image[0][valid_dwi], [1, 99])
axes[3].imshow(
    np.ma.masked_where(~valid_dwi, image[0]),
    cmap='gray',
    vmin=low,
    vmax=high,
)

axes[3].imshow(np.ma.masked_where(mask <= 0, mask), cmap='Reds', alpha=0.45, vmin=0, vmax=1)
axes[3].set_title(f"{sample['sample_id']} lesion overlay")
axes[3].axis('off')
plt.tight_layout()
plt.show()

## 7. Build memory bank trên cả hai GPU

In [ ]:
subprocess.run(
    [sys.executable, 'build_memory.py', '--config', 'configs/isles2022.yaml', '--base_config', 'configs/base_config.yaml'],
    cwd=REPO_DIR,
    check=True,
)

## 8. Train DataParallel và chọn best bằng validation mDice

In [ ]:
subprocess.run(
    [sys.executable, 'train.py', '--base_config', 'configs/base_config.yaml', '--data_config', 'configs/isles2022.yaml'],
    cwd=REPO_DIR,
    check=True,
)

`train.py` đánh giá validation mỗi epoch và lưu `checkpoints/hpma_isles-2022_best.pt` theo validation mDice; `*_last.pt` là checkpoint epoch cuối. Cell có `check=True`, nên notebook dừng ngay khi build-memory hoặc train thất bại

## 9. Đánh giá test

In [ ]:
BEST_CHECKPOINT = REPO_DIR / 'checkpoints' / 'hpma_isles-2022_best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Không có best checkpoint: {BEST_CHECKPOINT}")

subprocess.run(
    [
        sys.executable, 'eval.py',
        '--base_config', 'configs/base_config.yaml',
        '--data_config', 'configs/isles2022.yaml',
        '--checkpoint', 'checkpoints/hpma_isles-2022_best.pt',
        '--split', 'test',
    ],
    cwd=REPO_DIR,
    check=True,
)

## 10. Lưu và tải artifacts sau khi commit hoàn tất

Save & Run All chạy trong phiên riêng trên máy chủ Kaggle, không cần treo máy cá nhân. Khi chạy xong, `hpma_isles2022_artifacts.zip` và các checkpoint nằm trong tab **Output** của version. Kaggle giới hạn phiên notebook khoảng 12 giờ; tổng thời gian download, setup, train và eval phải nằm trong giới hạn đó.


In [ ]:
import shutil
from zipfile import ZIP_DEFLATED, ZipFile

# Training/evaluation are complete; raw scans can be downloaded again and are not model artifacts.
shutil.rmtree(DATASET_DIR, ignore_errors=True)

artifacts = [
    REPO_DIR / 'checkpoints' / 'hpma_isles-2022_best.pt',
    REPO_DIR / 'checkpoints' / 'hpma_isles-2022_last.pt',
    REPO_DIR / 'memory' / 'bank_cache' / 'isles2022_bank.pt',
    REPO_DIR / 'outputs' / 'training.log',
    REPO_DIR / 'configs' / 'base_config.yaml',
    REPO_DIR / 'configs' / 'isles2022.yaml',
]

missing = [path for path in artifacts[:4] if not path.is_file()]
if missing:
    raise FileNotFoundError('Thiếu training artifacts: ' + ', '.join(map(str, missing)))

for artifact in artifacts:
    print(f"{'OK' if artifact.is_file() else 'MISSING'}: {artifact}")

bundle = WORKING_DIR / 'hpma_isles2022_artifacts.zip'
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as archive:
    for artifact in artifacts:
        if artifact.is_file():
            archive.write(artifact, artifact.relative_to(REPO_DIR))

print(f"Artifact bundle ready: {bundle}")
print('Save & Run All chạy trên máy chủ Kaggle; bạn không cần giữ máy cá nhân mở. Khi commit hoàn tất, tải ZIP từ tab Output của version.')


In [ ]:
working_bytes = sum(path.stat().st_size for path in WORKING_DIR.rglob('*') if path.is_file() and not path.is_symlink() and not path.is_symlink())
print(f"/kaggle/working total: {working_bytes / 2**30:.2f} GiB")

In [ ]:
if working_bytes > 20 * 2**30:
    raise RuntimeError("Output vượt giới hạn 20 GB; dọn artifacts không cần thiết trước Save Version.")

print('Tạo Save Version và bật lưu outputs để giữ checkpoints trong Kaggle Notebook Output.')